[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/1-using-tool.ipynb)

## Colab setup

Run the next cell first on Google Colab. It sparse-checkouts this public repo and installs packages only when the kernel is Colab. Local Jupyter and VS Code skip that work.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) or in an environment cell: `os.environ["OPENAI_API_KEY"] = "sk-..."`. Do that before the lesson setup cell. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

In [ ]:
# Colab setup. Run this cell before the other code cells.
# Local Jupyter and VS Code skip the clone and the install.
#
# An API key is required. On Google Colab, set OPENAI_API_KEY in one of these ways:
#   * Secrets (the key icon): a secret named OPENAI_API_KEY
#   * an environment cell: os.environ["OPENAI_API_KEY"] = "sk-..."
# Locally, put the key in the repository-root .env.
# Do not commit a key. This cell does not print the value.
# Edits you make in Colab stay in the session. They do not push to GitHub.

import os
import sys
from pathlib import Path


def in_colab():
    """True on Google Colab. False in local Jupyter and VS Code."""
    try:
        import google.colab
    except ImportError:
        return False
    return Path("/content").is_dir() and google.colab is not None


if not in_colab():
    print("Not Colab. Skipped clone and pip install.")
else:
    import shutil
    import subprocess

    missing = []
    for module_name, requirement in (
        ("openai", "openai>=1.40.0"),
        ("dotenv", "python-dotenv>=1.0.1"),
        ("httpx", "httpx>=0.27.0"),
    ):
        try:
            __import__(module_name)
        except ImportError:
            missing.append(requirement)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

    repo = Path("/content/book-agents")
    marker = repo / "tutorial" / "common" / "client.py"
    if not marker.is_file():
        if repo.exists() and not (repo / ".git").exists():
            raise RuntimeError(
                str(repo) + " exists but is not a git checkout. "
                "Move that folder aside and run this cell again."
            )
        if repo.exists():
            shutil.rmtree(repo)
        url = "https://github.com/junlinghu/book-agents.git"
        try:
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                url, str(repo),
            ])
            subprocess.check_call([
                "git", "-C", str(repo), "sparse-checkout", "set", "tutorial",
            ])
        except subprocess.CalledProcessError:
            if repo.exists():
                shutil.rmtree(repo)
            subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo)])
        if not marker.is_file():
            raise RuntimeError(
                "Colab setup could not find tutorial/common/client.py after cloning."
            )

    os.chdir(repo)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        try:
            from google.colab import userdata
            secret = userdata.get("OPENAI_API_KEY")
        except Exception:
            secret = ""
        if secret and str(secret).strip():
            os.environ["OPENAI_API_KEY"] = str(secret).strip()

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        raise RuntimeError(
            "OPENAI_API_KEY is empty. On Colab, add a secret named OPENAI_API_KEY "
            "(the key icon) and rerun this cell. Locally, copy .env.example to .env "
            "and paste a key from https://platform.openai.com/api-keys. Never commit .env."
        )
    print("Colab: tutorial/ is ready. OPENAI_API_KEY is set (value hidden).")


# 1. Tool calling

You are building the counter concierge for Hearth Lane Café. This notebook is the first piece: one tool, two model calls.

The model must not invent the return rule. It asks for `get_shop_fact`. This process reads `docs/policy.md` and returns the Returns section. The second model call writes the answer from that string.

Later notebooks keep this tool and add more. Run this file from top to bottom. The model calls need `OPENAI_API_KEY`.

The code cells are the whole agent for this lesson. There is no earlier notebook to open first.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, the setup cell above reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `1-using-tool.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '1-using-tool'
LESSON_NUMBER = 1
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Test chat

One Chat Completions call, with no tools. This confirms `OPENAI_API_KEY` and the model before the tool-calling run.

In [ ]:
def test_chat(question):
    """One Chat Completions call with no tools. Prints the assistant answer.

    Uses the key and model already loaded by the setup cell.
    """
    require_key()
    turn = chat([{"role": "user", "content": question}], tools=[])
    print("ANSWER:", turn["content"])
    return turn


test_chat("Reply with exactly: ok")


## Shared helpers

Schema, message copying, and the check used by every later cell.

In [ ]:
import json
import re
import sqlite3
import time
from pathlib import Path

HERE = ROOT / "tutorial"
DOCS = HERE / "docs"
DATA = HERE / "data"
VAR = HERE / "var"

HANDLERS = {}
TOOLS = []


def register(name, description, properties, required, fn):
    """Add one Chat Completions tool. ``fn`` runs only if the harness allows it."""
    spec = {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }
    # Re-running a cell replaces the previous schema instead of stacking a second copy.
    TOOLS[:] = [tool for tool in TOOLS if tool["function"]["name"] != name]
    TOOLS.append(spec)
    HANDLERS[name] = fn


def preview(text, limit=500):
    """Shorten a tool result for the printed trace. The model still gets the full string."""
    text = text if isinstance(text, str) else str(text)
    if len(text) <= limit:
        return text
    return text[:limit] + "\n... [" + str(len(text)) + " characters]"


def assistant_message(turned):
    """Copy an assistant turn into the message list, including any tool calls."""
    message = {"role": "assistant", "content": turned["content"] or ""}
    calls = turned["tool_calls"] or []
    if calls:
        message["tool_calls"] = []
        for call in calls:
            raw = call.get("raw") or json.dumps(call["arguments"], sort_keys=True)
            message["tool_calls"].append({
                "id": call["id"],
                "type": "function",
                "function": {"name": call["name"], "arguments": raw},
            })
    return message


def check(condition, message):
    """Print whether a lesson expectation held. A miss warns and continues.

    A live model can phrase an answer differently. The warning names the
    miss without stopping the notebook.
    """
    if condition:
        print("check ok:", message)
        return
    print("check warning:", message)


def base_rules():
    return (
        "You are the counter concierge for Hearth Lane Café. "
        "Shop facts come from tools, not from guesses in the prompt. "
        "If a tool does not say, say you don't know. "
        "Cite the tool or the docs path you used. "
        "Do not invent a Wi-Fi password, a refund, or a shipping exception. "
        "Do not charge a card or send email yourself."
    )


def call_tool(call):
    """Run one handler. Bad arguments become an ERROR string, not a crash."""
    name = call["name"]
    args = call["arguments"]
    if not isinstance(args, dict):
        return "ERROR: arguments must be a JSON object."
    if args.get("_error"):
        return "ERROR: " + str(args["_error"])
    fn = HANDLERS.get(name)
    if fn is None:
        return "ERROR: unknown tool " + repr(name) + "."
    return fn(args)


## Tutorial 1 — one tool

`get_shop_fact` opens the shop document. The model only sees the string you return.

In [ ]:
from tutorial.common.tools import read_file as read_docs_file


def _read_docs(path):
    """Read one shop document. The path jail lives in ``tutorial.common.tools``."""
    if not isinstance(path, str):
        return "ERROR: path must be a string. Example: policy.md"
    return read_docs_file(str(DOCS), path)


def _section(markdown, heading):
    lines = markdown.splitlines()
    start = None
    prefix = "## " + heading
    for index, line in enumerate(lines):
        if line.strip() == prefix:
            start = index + 1
            break
    if start is None:
        return ""
    body = []
    for line in lines[start:]:
        if line.startswith("## "):
            break
        body.append(line)
    return "\n".join(body).strip()


def get_shop_fact(args):
    """Return one policy or FAQ section. The model names a topic; this function opens the file."""
    topic = str(args.get("topic", "")).strip().lower()
    mapping = {
        "returns": ("policy.md", "Returns"),
        "shipping": ("policy.md", "Shipping"),
        "hours": ("faq.md", "Where and when"),
        "allergens": ("faq.md", "Allergens"),
    }
    if topic not in mapping:
        return "ERROR: topic must be returns, shipping, hours, or allergens."
    path, heading = mapping[topic]
    document = _read_docs(path)
    if document.startswith("ERROR:"):
        return document
    body = _section(document, heading)
    if not body:
        return "ERROR: " + heading + " is missing from " + path + "."
    return "TOPIC: " + topic + "\nSOURCE: docs/" + path + "\n\n" + body


register(
    "get_shop_fact",
    "Look up one Hearth Lane rule. topic is returns, shipping, hours, or allergens. "
    "The result quotes docs/policy.md or docs/faq.md.",
    {"topic": {"type": "string", "description": "returns, shipping, hours, or allergens"}},
    ["topic"],
    get_shop_fact,
)


## System prompt

The prompt stays short. Documents, notes, skills, and pages arrive through tools.

In [ ]:
def system_text():
    parts = [base_rules()]
    parts.append("Use get_shop_fact before you state a return, shipping, hours, or allergen rule.")
    return "\n\n".join(parts)


## Run

Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
print("registered tools:", [tool["function"]["name"] for tool in TOOLS])
print()

question = "Can a customer return an opened bag of house coffee?"
messages = [
    {"role": "system", "content": system_text()},
    {"role": "user", "content": question},
]
print("QUESTION:", question)
print()

# The model proposes a tool call. This process executes it. The model does not open the file.
first = chat(messages, TOOLS)
messages.append(assistant_message(first))
names = [call["name"] for call in first["tool_calls"]]
print("tool_calls:", names)
check("get_shop_fact" in names, "the model asks for get_shop_fact")

if not first["tool_calls"]:
    print("ANSWER:", first["content"])
    print("stop: final (no tool call)")
    check(False, "the model asked for a tool before answering")
else:
    for call in first["tool_calls"]:
        result = call_tool(call)
        print("[tool]", call["name"], json.dumps(call["arguments"], sort_keys=True))
        print(preview(result))
        print()
        messages.append({
            "role": "tool",
            "tool_call_id": call["id"],
            "content": result,
        })
    second = chat(messages, TOOLS)
    print("ANSWER:", second["content"])
    print("stop: final")
    check(not second["tool_calls"], "the second turn is a final answer")
    check("final sale" in (second["content"] or "").lower(), "answer says opened coffee is final sale")
